In [ ]:
# Cell 1 — Install
!pip install -q "openenv-core[core]>=0.2.1"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers datasets accelerate peft
!pip install -q torch-geometric
!git clone https://github.com/Godhand-Arnav/Scalar-finals.git /content/forge-rl

import sys, os, torch
sys.path.insert(0, '/content/forge-rl')
os.chdir('/content/forge-rl')

if not torch.cuda.is_available():
    raise RuntimeError('NO GPU DETECTED. Runtime -> Change runtime type -> T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete.')

In [ ]:
# Cell 2 — Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
print('Model loaded.')

In [ ]:
# Cell 3 — Reward function
import numpy as np
from env.forge_env import ForgeEnv, ForgeEnvConfig

# 0=query_source, 3=temporal_audit, 1=cross_reference (5=context_retrieve, NOT temporal)
TOOL_ACTIONS = [0, 3, 1]

# Action space: 10=misinfo, 11=satire, 12=verified
# fabricated and out_of_context → closest is misinfo (10)
def _parse_verdict(text: str) -> int:
    t = text.lower()
    # Check satire first (distinct signal words)
    if any(w in t for w in ['satire', 'parody', 'joke', 'humor', 'comedic']):
        return 11
    # Check verified (must come before 'false' check to avoid substring collision)
    if any(w in t for w in ['verified', 'legitimate', 'accurate', 'credible', 'true claim']):
        return 12
    # Misinfo umbrella (fabricated, out_of_context, mislead all → misinfo)
    if any(w in t for w in ['misinfo', 'false', 'manipulat', 'mislead',
                             'fabricat', 'out of context', 'deceptive', 'disinformation']):
        return 10
    return 10  # safe default: misinfo (a terminal verdict, not a tool)

def _safe_step(env, action):
    result = env.step(action)
    if isinstance(result, tuple):
        if len(result) == 5:
            _, reward, terminated, truncated, _ = result
            return float(reward), bool(terminated or truncated)
        elif len(result) == 4:
            _, reward, terminated, _ = result
            return float(reward), bool(terminated)
        elif len(result) == 2:
            _, reward = result
            return float(reward), False
    return float(getattr(result, 'reward', 0.0)), bool(getattr(result, 'done', False))

def _safe_reset(env):
    result = env.reset()
    if isinstance(result, tuple) and len(result) == 2:
        return result
    return result, {}

def reward_fn(prompts, completions, claim_texts=None, **kwargs):
    # GRPOTrainer passes dataset cols via kwargs — also check kwargs
    if claim_texts is None:
        claim_texts = kwargs.get('claim_texts', None)
    rewards = []
    for i, completion in enumerate(completions):
        comp_text = (completion[-1]['content'] if isinstance(completion, list) else str(completion))
        try:
            cfg = ForgeEnvConfig(budget=6, seed=42 + i)
            env = ForgeEnv(cfg)
            obs, info = _safe_reset(env)
            if claim_texts and i < len(claim_texts):
                for attr in ('_claim_text', '_claim_text_initial'):
                    if hasattr(env, attr): setattr(env, attr, claim_texts[i])
            done = False
            for act in TOOL_ACTIONS:
                if done:
                    break
                try:
                    _, done = _safe_step(env, act)
                except Exception:
                    break  # budget exhausted or env error — stop tools
            
            # Only submit verdict if episode not already done
            if not done:
                verdict_action = _parse_verdict(comp_text)
                try:
                    reward, _ = _safe_step(env, verdict_action)
                except Exception:
                    reward = 0.001
            else:
                # Env terminated before verdict — penalise lightly
                reward = 0.001
            rewards.append(float(np.clip(reward, 0.001, 0.999)))
        except Exception as e:
            print(f'[reward_fn] episode {i} error: {e}')
            rewards.append(0.001)
    return rewards

print('reward_fn ready.')

In [ ]:
# Cell 4 — Dataset
from datasets import Dataset
import random

PROMPT_TMPL = ('You are a misinformation forensics agent. Investigate the claim:\n'
               'CLAIM: {claim}\n\n'
               'Submit your verdict:\n'
               '- "This is MISINFO because [reason]"\n'
               '- "This is SATIRE because [reason]"\n'
               '- "This is VERIFIED because [reason]"\n'
               '- "This is FABRICATED because [reason]"\n'
               '- "This is OUT OF CONTEXT because [reason]"\n'
               'Your verdict:')

TASK_NAMES = ['fabricated_stats', 'out_of_context', 'coordinated_campaign',
              'satire_news', 'verified_fact', 'politifact_liar',
              'image_forensics', 'sec_fraud']

def _get_claim(task, seed):
    try:
        cfg = ForgeEnvConfig(budget=3, seed=seed)
        env = ForgeEnv(cfg)
        _, info = _safe_reset(env)
        for attr in ('_claim_text', 'claim_text'):
            val = getattr(env, attr, None)
            if val: return str(val)
        if info and 'claim_text' in info: return info['claim_text']
    except: pass
    return f'Unverified claim #{seed} in domain: {task}'

random.seed(42)
claims, prompts = [], []
for i in range(200):
    claim = _get_claim(TASK_NAMES[i % len(TASK_NAMES)], seed=i)
    claims.append(claim)
    prompts.append([{'role': 'user', 'content': PROMPT_TMPL.format(claim=claim)}])

dataset = Dataset.from_dict({'prompt': prompts, 'claim_texts': claims})
print(f'Dataset: {len(dataset)} rows')

In [ ]:
# Cell 5 — Baseline
def evaluate_heuristic(n_episodes=20, default_verdict=10):  # 10=misinfo, was 9=tool
    rewards, correct = [], 0
    for i in range(n_episodes):
        try:
            env = ForgeEnv(ForgeEnvConfig(budget=6, seed=100 + i))
            _safe_reset(env)
            done = False
            for act in TOOL_ACTIONS:
                if done:
                    break
                try:
                    _, done = _safe_step(env, act)
                except Exception:
                    break
            if not done:
                try:
                    reward, _ = _safe_step(env, default_verdict)
                except Exception:
                    reward = 0.001
            else:
                reward = 0.001
            rewards.append(float(reward))
            if reward > 0.5: correct += 1
        except Exception as e:
            print(f'Eval error ep {i}: {e}')
            rewards.append(0.001)
    return float(np.mean(rewards)), correct / n_episodes

baseline_reward, baseline_acc = evaluate_heuristic()
print(f'HEURISTIC BASELINE — reward: {baseline_reward:.4f} | acc: {baseline_acc:.2%}')

In [ ]:
# Cell 6 — GRPO Training
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='./forge-grpo',
    max_steps=150,
    num_generations=4,
    max_completion_length=128,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    save_steps=50,
    logging_steps=5,
    report_to='none',
    warmup_ratio=0.1,
)

try:
    trainer = GRPOTrainer(model=model, reward_funcs=reward_fn, args=config,
                          train_dataset=dataset, processing_class=tokenizer)
except TypeError:
    trainer = GRPOTrainer(model=model, reward_funcs=reward_fn, args=config,
                          train_dataset=dataset, tokenizer=tokenizer)

print('Starting GRPO training (max 150 steps ~45 min on T4)...')
trainer.train()
print('Training complete.')

In [ ]:
# Cell 7 — Evaluation + plots
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
import os
os.makedirs('results', exist_ok=True)

def evaluate_trained(n_episodes=20):
    rewards, correct = [], 0
    FastLanguageModel.for_inference(model)
    for i in range(n_episodes):
        try:
            claim = _get_claim(TASK_NAMES[i % len(TASK_NAMES)], seed=200 + i)
            prompt_ids = tokenizer.apply_chat_template(
                [{'role': 'user', 'content': PROMPT_TMPL.format(claim=claim)}],
                tokenize=True, return_tensors='pt', add_generation_prompt=True,
            ).to('cuda')
            outputs = model.generate(input_ids=prompt_ids, max_new_tokens=80,
                                     temperature=0.3, do_sample=True)
            response = tokenizer.decode(outputs[0][prompt_ids.shape[1]:], skip_special_tokens=True)
            env = ForgeEnv(ForgeEnvConfig(budget=6, seed=200 + i))
            _safe_reset(env)
            done = False
            for act in TOOL_ACTIONS:
                if done:
                    break
                try:
                    _, done = _safe_step(env, act)
                except Exception:
                    break
            if not done:
                try:
                    reward, _ = _safe_step(env, _parse_verdict(response))
                except Exception:
                    reward = 0.001
            else:
                reward = 0.001
            rewards.append(float(reward))
            if reward > 0.5: correct += 1
        except Exception as e:
            print(f'Trained eval error ep {i}: {e}')
            rewards.append(0.001)
    return float(np.mean(rewards)), correct / n_episodes

trained_reward, trained_acc = evaluate_trained()
print(f'BASELINE — {baseline_reward:.4f} | {baseline_acc:.2%}')
print(f'TRAINED  — {trained_reward:.4f} | {trained_acc:.2%}')
print(f'DELTA    — {trained_reward - baseline_reward:+.4f} | {trained_acc - baseline_acc:+.2%}')

steps, rew = [], []
REWARD_LOG_KEYS = (
    'rewards/mean', 'reward/mean', 'reward',
    'train/reward', 'rewards/reward_mean',   # TRL version variants
    'mean_reward',
)
for l in trainer.state.log_history:
    if 'step' not in l:
        continue
    for key in REWARD_LOG_KEYS:
        if key in l:
            steps.append(l['step'])
            rew.append(l[key])
            break

fig, ax = plt.subplots(figsize=(10, 5))
if steps: ax.plot(steps, rew, 'b-', linewidth=2.5, label='Training reward')
ax.axhline(y=baseline_reward, color='r', linestyle='--', linewidth=2,
           label=f'Heuristic baseline ({baseline_reward:.3f})')
ax.set_xlabel('Training Step'); ax.set_ylabel('Mean Episode Reward')
ax.set_title('FORGE-MA: LLM Agent Learning Misinformation Detection via GRPO')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('results/reward_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: results/reward_curve.png')